# Notebook 10: Citation Dynamics
We examine how citations differ **over time** between sharing status (*sharing vs. non-sharing*) and across sharing granularities (*fixation-* vs. *trial-* vs. *participant-level*).<br>
We calculate the number of citations for each article in every calendar year post-publication (ignoring "year 0", the publication year itself, and every calendar year after the last complete one, 2025, due to partial year effects). We then examine the differences in year-by-year citations across sharing categories while accounting for other article-level covariates (e.g., U.S. affiliation, venue impact score, etc.). We do so by fitting a mixed-effects model regressing *Citations in year `k` post publication* on the article covariates, including a `C(Is Sharing) * k` interaction term to test whether the citations differ between *sharing* and *non-sharing* over time (i.e. the citation **dynamics**). The same is to be done for the subset of sharing articles, testing whether a `C(Sharing Class) * k` interaction term is significant, which would indicate different citation dynamics for different shared granularities (not implemented yet).
<br><br>
**Notes:**
1. We use citation *flow* (count per year) rather than *stock* (cumulative count). A cumulative count contains all of the earlier years and can only increase, so its successive values are strongly autocorrelated, far beyond what a random intercept per article can absorb. The flow in each year is also correlated within an article, but a random intercept captures that.
2. `k` is discrete but equally spaced, so we treat it as a *continuous* variable (not a *categorical* one) in the mixed-effects model. This reduces the number of parameters, increases statistical power, and means the interaction term directly gives us the difference in slope between sharing and non-sharing articles (or between sharing granularities) over time.
3. In every model we standardize `k` (`k_z`) and `log(k)` (`log_k_z`) to mean `0` and `SD=1`, which improves model convergence and the interpretability of the coefficients: the sharing-by-time coefficient is per SD of `k` (or of `log(k)`), and the main effect of sharing is the group difference at the mean `k`, an average gap and not the dynamics. The publication year (`k = 0`) is excluded from the data, so the dynamics are tested by the sharing-by-time interaction.
4. The sharing-by-time test is two-sided: unlike the sharing hypothesis in the other notebooks, we have no directional hypothesis about how the groups diverge over time.

### Outline
1. **Data:** how many articles contribute to each year after publication, and what the observed yearly citations look like.
2. **Main analysis:** four candidate shapes of time, $g(k)$, are fitted with the a-priori Gaussian on $log(1 + \text{Citations})$ and the maximal random-effects structure; the one with the lowest AIC is the winner, and we interpret its sharing-by-time interaction.
3. **Robustness #1 (the outcome):** the winner is refitted with other offsets in $log(y + c)$ and without the last, thin, year.
4. **Robustness #2 (the family):** the winner (and a near-tied $g(k)$, if there is one) is refitted as a Gaussian on the raw counts, a Poisson and a negative binomial.
5. **Visualization:** the winner's predicted citations per year for sharing and non-sharing articles.

**Reading rules.** The winner has the lowest AIC; candidates within 2 AIC of it are near-ties. Coefficients are per SD of the standardized time term, so the four candidates (whose interactions are on different terms) are compared on the *sign and significance* of the sharing-by-time coefficient, not on its size.

### Model Specification
We use R's `lme4` package to fit a family of LMEs, with the following specification:
$$ f(\text{Citations}_{i,k}) \sim \text{Is Sharing} \times k + g(k) + \text{Has US Author} + \text{Is OA} + \text{Has Preprint} + log(\text{Number of Authors}) + \text{Impact Score} + C(\text{Publication Year}) + (1 | \text{Venue}) + (1 + k | \text{Venue}:\text{Article}) $$

Here, $f()$ stands for how the citation count enters each of four models: a Gaussian on $log(1 + \text{Citations})$, a Gaussian on the raw counts, and Poisson and negative-binomial models, which model the mean count through a $log$ link instead of transforming the outcome. We use these four to check that the conclusion does not depend on the distributional assumption for skewed, zero-heavy yearly counts.
<br><br>
We use the Gaussian on $log(1 + \text{Citations})$ as the primary model, chosen a priori. `Thelwall & Wilson (2014)` found that citation counts are well described by a discrete lognormal distribution, and recommended using the $log(1 + y)$ transformation as the best stand-in available for GLMMs (since the discrete lognormal itself has no mixed-model implementation). They verified this claim using simulations that cover the regime of our data (our yearly flow fits a discrete lognormal with $\mu = 0.63$, $\sigma = 1.19$); and found that $log(1 + y)$ keeps the nominal 5% false-positive rate and is the most powerful trustworthy option, while negative-binomial regression falsely flags factors about 2-3× too often, worsening with sample size.<br>
One caveat is that their simulations were for single-level models, not for repeated measures.
<br><br>
The other three families are used as robustness checks, to be mentioned in the discussion section of the manuscript. They do not estimate the same quantity: the Gaussian on $log(1 + y)$ estimates a proportional change in the geometric mean of $1 + y$, while Poisson and negative binomial estimate a rate ratio of the arithmetic mean. We therefore compare the families on the sign and significance of the sharing-by-time coefficient, not on its size.
<br><br>
The $g(k)$ term is a (possibly non-linear) function of $k$ to account for non-linear effects of time on citations. We compare four candidates (linear, $log(k)$, $k + log(k)$ and $k + k^2$, described below), and the one with the lowest AIC wins. Candidates within 2 AIC of the winner (the usual margin for "equivalent" fits) are reported as near-ties and carried into the robustness checks.
<br><br>
We start from the maximal random-effects structure, $(1 | \text{Venue}) + (1 + k | \text{Venue}:\text{Article})$, to account for the nested structure of the data: we have many observations of the same article (over different years); and articles are in turn nested in venues. The random intercepts for articles account for the correlation between an article's repeated yearly counts, while the random intercepts for venues account for the correlation between articles in the same venue. The random slope of $k$ lets each article follow its own time trend (for the $log(k)$ candidate, the slope is on $log(k)$).<br>
If a model does not converge, or returns a singular fit (a variance at zero or a correlation of $\pm 1$), with this structure, we refit it with random intercepts only, $(1 | \text{Venue} / \text{Article})$. The structure each model used is shown in every table.
<br><br>
**Limitations of this structure:**
- Sharing varies between articles and `k` within them, so omitting the random slope of `k` can make the SE of the sharing-by-time interaction too small. The maximal structure protects against that, but a model that falls back to random intercepts loses the protection (the structure used is always shown). Even with the slope, a random intercept and slope imply a particular covariance between an article's years; serial correlation within an article is not modelled.
- Publication year enters as a categorical fixed effect, and calendar year is publication year plus `k`, so any linear calendar-year trend is absorbed into the `k` slope. Later cohorts also contribute fewer years, so the high-`k` tail is dominated by older articles with few sharers (in the superseded analysis, N = 234 at year 1 fell to 84 by year 7, with 19 sharers left). The tail is thin, so the figure shows the number of articles behind every point instead of hiding the late years, and a sensitivity check below drops the last year.


In [ ]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure
from helpers.r_models import GlmmResult, fit_glmm

pio.renderers.default = "browser"

## Setup

In [ ]:
ALPHA = 0.05
PRIMARY_FAMILY, PRIMARY_OUTCOME = "gaussian", "log_citations"       # chosen a priori (see the specification above)

# the candidate g(k) terms: the fixed time terms (the sharing-by-time interaction is on `k_z` or `log_k_z`), the same terms
# without the interaction (for the likelihood-ratio test), and the time term of the article-level random slope
CANDIDATES = {
    "k": dict(time="is_sharing_data * k_z", no_interaction="is_sharing_data + k_z", slope="k_z"),
    "log(k)": dict(time="is_sharing_data * log_k_z", no_interaction="is_sharing_data + log_k_z", slope="log_k_z"),
    "k + log(k)": dict(time="is_sharing_data * k_z + log_k_z", no_interaction="is_sharing_data + k_z + log_k_z", slope="k_z"),
    "k + k^2": dict(time="is_sharing_data * k_z + I(k_z^2)", no_interaction="is_sharing_data + k_z + I(k_z^2)", slope="k_z"),
}
COVARIATES = "has_us_author + is_open_access + has_preprint + log_number_of_authors + venue_impact + factor(publication_year)"
RANDOM_EFFECTS = {      # the maximal structure first, random intercepts only as the fallback
    "full": "(1 | venue_name) + (1 + {slope} | venue_name:article)",
    "intercept": "(1 | venue_name/article)",
}


def build_formula(outcome: str, candidate: str, structure: str, interaction: bool = True) -> str:
    spec = CANDIDATES[candidate]
    time_terms = spec["time"] if interaction else spec["no_interaction"]
    return f"{outcome} ~ {time_terms} + {COVARIATES} + {RANDOM_EFFECTS[structure].format(slope=spec['slope'])}"

In [ ]:
def is_usable(fit: GlmmResult) -> bool:
    return fit.fit["converged"] and not fit.fit["singular"]


def fit_with_fallback(
        data: pd.DataFrame, outcome: str, candidate: str, family: str,
        structures: tuple[str, ...] = ("full", "intercept"), **fit_kwargs,
) -> tuple[GlmmResult, str]:
    """Fit `candidate` with each random structure in turn; return the first usable fit (converged, not singular)
    and its structure, or the last structure's fit when none is usable."""
    for structure in structures[:-1]:
        fit = fit_glmm(data, build_formula(outcome, candidate, structure), family, **fit_kwargs)
        if is_usable(fit):
            return fit, structure
    return fit_glmm(data, build_formula(outcome, candidate, structures[-1]), family, **fit_kwargs), structures[-1]


def sharing_by_time(fit: GlmmResult) -> pd.Series:
    """The sharing-by-time coefficient: the only `is_sharing_data:` term in these models."""
    return fit.fixed_effects.filter(like="is_sharing_data:", axis=0).iloc[0]


def comparison_table(fits: dict[str, GlmmResult], structures: dict[str, str]) -> pd.DataFrame:
    """AIC, the gap to the best model, Akaike weights and the sharing-by-time coefficient of each model, best first."""
    rows = {
        name: {
            "structure": structures[name], "converged": fit.fit["converged"], "singular": fit.fit["singular"], "AIC": fit.fit["aic"],
            **sharing_by_time(fit)[["estimate", "se", "p"]].astype(float).to_dict(),
        }
        for name, fit in fits.items()
    }
    table = pd.DataFrame.from_dict(rows, orient="index")
    table["delta AIC"] = table["AIC"] - table["AIC"].min()
    weights = np.exp(-table["delta AIC"] / 2)
    table["weight"] = weights / weights.sum()
    return table.sort_values("AIC")


def effect_table(fits: dict[str, tuple[GlmmResult, str]], reference_sign: float) -> pd.DataFrame:
    """The sharing-by-time coefficient of each refit, and whether it keeps the reference sign and significance."""
    rows = {}
    for name, (fit, structure) in fits.items():
        effect = sharing_by_time(fit)
        rows[name] = {
            "structure": structure, "converged": fit.fit["converged"], "singular": fit.fit["singular"], "N": fit.fit["n_obs"],
            "b": effect["estimate"], "SE": effect["se"], "p": effect["p"], "|t|": abs(effect["estimate"] / effect["se"]),
            "rate ratio": effect.get("rate_ratio", np.nan),
            "same sign": bool(np.sign(effect["estimate"]) == reference_sign), "significant": bool(effect["p"] < ALPHA),
        }
    return pd.DataFrame.from_dict(rows, orient="index")

In [3]:
def citations_per_year(
        articles: pd.DataFrame, grouping_col: str, cumulative: bool = False
) -> pd.DataFrame:
    """Citations per year since publication, indexed by (article, group).

    The reshape from absolute calendar years to years-since-publication is done by
    `dataset.citations_since_publication()`. This wrapper only applies the two analysis choices
    belonging to this notebook, which that helper deliberately leaves to its callers: drop the
    publication year itself (`year_0`, partial for every article), and drop any calendar year that
    has not fully elapsed for the whole sample.

    The incomplete-year cutoff comes from `dataset.citations_through_shared_year()` rather than a
    hard-coded year, so it tracks the data instead of going stale at the next re-fetch.

    :param articles: frame carrying the `Citations20XX` columns, `PublicationYear`, `LastUpdate`
        and `grouping_col`.
    :param grouping_col: column to append as the second index level, e.g. `is_sharing_data`.
    :param cumulative: True for cumulative citations (stock), False for per-year counts (flow).
    :return: frame indexed by (article, `grouping_col`), one column per year offset.
    """
    _, shared_year = dataset.citations_through_shared_year(articles)
    incomplete = [
        col for col in articles.columns
        if col.startswith("Citations") and col.removeprefix("Citations").isdigit()
        and int(col.removeprefix("Citations")) > shared_year
    ]
    per_year = dataset.citations_since_publication(articles.drop(columns=incomplete))
    per_year = per_year.drop(columns="year_0")
    per_year.columns = [int(col.removeprefix("year_")) for col in per_year.columns]
    if cumulative:
        per_year = per_year.cumsum(axis=1)
    return per_year.set_index(articles[grouping_col], append=True)

### Load Data

In [4]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


## Sharing vs. Non-Sharing

### The Data
One row per article-year. Before any model, how many articles contribute to each year after publication, and what do the observed yearly citations look like? Only the oldest cohort reaches the last years, so the tail is thin, especially for sharing articles.

In [ ]:
GROUPING_VARIABLE = "is_sharing_data"

In [6]:
per_year = (
    citations_per_year(combined, grouping_col=GROUPING_VARIABLE, cumulative=False)
    .droplevel(GROUPING_VARIABLE)
    .stack()
    .rename("citations")
    .dropna()                               # years an article has not reached yet
)
per_year.index.names = ["article", "k"]     # k is the year offset since publication

# long format: one row per article-year, with the article's covariates repeated across its rows
# (publication year is OpenAlex's `PublicationYear`, not Godwin et al.'s `YEAR_PUBLISHED`)
article_covariates = dataset.to_smf_columns(FEATURES_DF).assign(publication_year=combined["PublicationYear"].astype(int))
DYNAMICS_DF = (
    per_year.reset_index()
    .merge(article_covariates, left_on="article", right_index=True, validate="many_to_one")
)
DYNAMICS_DF

,article,k,citations,is_sharing_data,sharing_class,has_us_author,is_open_access,has_preprint,log_weeks_since_pub,log_number_of_authors,venue_impact,venue_name,publication_year
0,0,1,2.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
1,0,2,3.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
2,0,3,0.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
3,0,4,1.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
4,0,5,1.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1342,250,8,2.0,0,NONE,1,1,0,6.134018,0.693147,0.429831,Journal of Vision,2017
1343,251,1,2.0,0,NONE,1,1,0,5.463226,1.098612,3.469027,Cognitive Research Principles and Implications,2021
1344,251,2,3.0,0,NONE,1,1,0,5.463226,1.098612,3.469027,Cognitive Research Principles and Implications,2021
1345,251,3,2.0,0,NONE,1,1,0,5.463226,1.098612,3.469027,Cognitive Research Principles and Implications,2021


In [ ]:
K_MEAN, K_SD = DYNAMICS_DF["k"].mean(), DYNAMICS_DF["k"].std()
LOG_K_MEAN, LOG_K_SD = np.log(DYNAMICS_DF["k"]).mean(), np.log(DYNAMICS_DF["k"]).std()

MODEL_DATA = DYNAMICS_DF.assign(
    k_z=(DYNAMICS_DF["k"] - K_MEAN) / K_SD,
    log_k_z=(np.log(DYNAMICS_DF["k"]) - LOG_K_MEAN) / LOG_K_SD,
    log_citations=np.log1p(DYNAMICS_DF["citations"]),
)

In [ ]:
# articles observed in each year after publication, and the geometric mean of their yearly citations (of 1 + citations, minus 1)
sample_sizes = MODEL_DATA.groupby(["is_sharing_data", "k"]).size()
observed_means = np.expm1(MODEL_DATA.groupby(["is_sharing_data", "k"])["log_citations"].mean())
pd.concat({"articles (n)": sample_sizes.unstack(0), "geometric mean citations": observed_means.unstack(0)}, axis=1).rename(
    columns={0: "not sharing", 1: "sharing"}, level=1
)

### Main Analysis
The a-priori model: a Gaussian on $log(1 + \text{Citations})$ with the maximal random-effects structure. We compare four candidate shapes of time, $g(k)$, all on standardized `k` (`k_z`) and `log(k)` (`log_k_z`):
- **`k`:** sharing $\times$ `k_z`. A constant difference in the yearly slope.
- **`log(k)`:** sharing $\times$ `log_k_z`. A difference that is largest early and levels off.
- **`k + log(k)`:** sharing $\times$ `k_z`, plus a `log_k_z` term shared by both groups. A linear sharing-by-time difference on top of a curved common trend.
- **`k + k^2`:** sharing $\times$ `k_z`, plus a `k_z^2` term shared by both groups. The same, with a quadratic common trend.

In the last two, the curvature is common to both groups, so the sharing difference stays linear in time and one coefficient summarizes it.

#### Typical Article
The predictions are for a typical article: the covariates at their mean (or mode, for the publication year) and the random effects at zero, for every year in the data.

In [ ]:
# typical-article predictions (covariates at their mean / mode, random effects at zero), for every year in the data:
# the late years rest on few articles, which the figure shows through its n labels
PREDICTION_GRID = pd.DataFrame(
    [{"is_sharing_data": sharing, "k": k} for sharing in (0, 1) for k in sorted(DYNAMICS_DF["k"].unique())]
)
for covariate in ["has_us_author", "is_open_access", "has_preprint", "log_number_of_authors", "venue_impact"]:
    PREDICTION_GRID[covariate] = DYNAMICS_DF[covariate].mean()
PREDICTION_GRID["publication_year"] = DYNAMICS_DF["publication_year"].mode()[0]
PREDICTION_GRID["k_z"] = (PREDICTION_GRID["k"] - K_MEAN) / K_SD
PREDICTION_GRID["log_k_z"] = (np.log(PREDICTION_GRID["k"]) - LOG_K_MEAN) / LOG_K_SD

#### Fit Models
Each candidate is fitted under both random structures, by maximum likelihood (so that AICs are comparable across different fixed effects). A candidate keeps its full-structure fit if that converged without a singular fit; otherwise its random-intercepts fit is used.

In [ ]:
full_fits = {
    name: fit_glmm(MODEL_DATA, build_formula(PRIMARY_OUTCOME, name, "full"), PRIMARY_FAMILY, newdata=PREDICTION_GRID)
    for name in CANDIDATES
}
intercept_fits = {
    name: fit_glmm(MODEL_DATA, build_formula(PRIMARY_OUTCOME, name, "intercept"), PRIMARY_FAMILY, newdata=PREDICTION_GRID)
    for name in CANDIDATES
}

# the full structure where it converged without a singular fit, the random-intercepts fit otherwise
selected_structure = {name: "full" if is_usable(full_fits[name]) else "intercept" for name in CANDIDATES}
selected_fits = {name: (full_fits if structure == "full" else intercept_fits)[name] for name, structure in selected_structure.items()}

#### Model Selection
We rank the candidates by AIC. `delta AIC` is the gap to the winner, and the Akaike weight is each model's relative support within this set. Candidates within 2 AIC of the winner are near-ties. Below the selection, all four candidates are shown under random intercepts only as a like-for-like reference (the selection table can mix random structures), with the AIC gained by the random slope (`full - intercept`; negative means the slope helps).

In [ ]:
selection = comparison_table(selected_fits, selected_structure)

winner, runner_up = selection.index[0], selection.index[1]
competitive = list(selection.index[selection["delta AIC"] <= 2])        # the winner and every candidate within 2 AIC of it
print(f"Chosen g(k): {winner} ({selection.loc[winner, 'structure']} random structure, Akaike weight {selection.loc[winner, 'weight']:.2f})")
print(f"Runner-up: {runner_up}, {selection.loc[runner_up, 'delta AIC']:.1f} AIC behind")
if len(competitive) > 1:
    print(f"NEAR-TIE: within 2 AIC of the winner: {', '.join(competitive[1:])}")
else:
    print("Clear winner: no other candidate is within 2 AIC")
if selection.loc[[winner, runner_up], "structure"].nunique() > 1:
    print("Note: the winner and the runner-up use different random structures, so this gap mixes g(k) and the random structure")

selection

In [ ]:
reference = comparison_table(intercept_fits, {name: "intercept" for name in CANDIDATES})
reference["full - intercept AIC"] = {name: full_fits[name].fit["aic"] - intercept_fits[name].fit["aic"] for name in CANDIDATES}
reference["full usable"] = {name: is_usable(full_fits[name]) for name in CANDIDATES}
reference

#### Interpretation
We look at how many candidates agree with the winner on the sign and significance of the sharing-by-time interaction, and at the winner's coefficient with its SE, CI and p, checked with a likelihood-ratio test of the interaction. The winner's p-value does not account for having been picked from four candidates, and the candidates are fitted to the same data, so the agreement across them is a consistency check, not independent evidence.

In [ ]:
WINNER = winner
WINNER_STRUCTURE = selection.loc[WINNER, "structure"]
winner_fit = selected_fits[WINNER]
print(f"The {WINNER} model ({WINNER_STRUCTURE} random structure) converged: {winner_fit.fit['converged']}, singular: {winner_fit.fit['singular']}")

# agreement across the candidates (coefficients are on different terms, so only the sign and significance are compared)
winner_sign = np.sign(selection.loc[WINNER, "estimate"])
agree = (np.sign(selection["estimate"]) == winner_sign) & (selection["p"] < ALPHA)
print(f"{agree.sum()} / {len(selection)} candidates have a significant sharing-by-time interaction with the winner's sign ({'negative' if winner_sign < 0 else 'positive'})")

# the winner's coefficient, and a likelihood-ratio test against the same model without the interaction (both ML)
winner_effect = sharing_by_time(winner_fit)
b, se = winner_effect["estimate"], winner_effect["se"]
z = stats.norm.ppf(0.975)
null_fit = fit_glmm(MODEL_DATA, build_formula(PRIMARY_OUTCOME, WINNER, WINNER_STRUCTURE, interaction=False), PRIMARY_FAMILY)
lrt_statistic = 2 * (winner_fit.fit["log_lik"] - null_fit.fit["log_lik"])
winner_summary = pd.Series({
    "b (per SD)": b, "SE": se, "CI low": b - z * se, "CI high": b + z * se, "p": winner_effect["p"],
    "LRT chi2 (1 df)": lrt_statistic, "LRT p": stats.chi2.sf(lrt_statistic, df=1),
})
if winner_effect.name == "is_sharing_data:k_z":      # the interaction is on linear k, so it has a per-year reading
    winner_summary["b (per year)"] = b / K_SD
    winner_summary["% change per year"] = 100 * np.expm1(b / K_SD)
else:
    print("The interaction is on log(k), so there is no per-year reading: compare the sign and significance only")
display(winner_summary)

print("Fixed Effects:")
display(winner_fit.fixed_effects)

print("Random Effects:")
display(winner_fit.variance_components)

### Robustness Check #1: The Outcome
#### Sensitivity to the Offset in `log(y + c)`
We refit the winner with different offsets and with the zeros dropped (no offset at all), since the offset of 1 is arbitrary. A refit passes if the sharing-by-time coefficient keeps the winner's sign and stays significant.<br>
The coefficients are on different scales for different offsets, so we compare them by `|t|` rather than by the estimate itself.<br>
Dropping the zeros also changes the sample and the question: it conditions on the article being cited that year, so it checks the direction and strength of the effect among cited article-years, not the same quantity.

In [ ]:
WINNER_STRUCTURES = ("full", "intercept") if WINNER_STRUCTURE == "full" else ("intercept",)     # each refit may fall back too

offset_fits = {"log(y + 1) (the winner)": (winner_fit, WINNER_STRUCTURE)}
for offset in (0.5, 0.25, 0.1):
    offset_fits[f"log(y + {offset})"] = fit_with_fallback(
        MODEL_DATA.assign(outcome=np.log(MODEL_DATA["citations"] + offset)), "outcome", WINNER, "gaussian", WINNER_STRUCTURES,
    )
positive = MODEL_DATA[MODEL_DATA["citations"] > 0]
offset_fits["log(y), zeros dropped"] = fit_with_fallback(
    positive.assign(outcome=np.log(positive["citations"])), "outcome", WINNER, "gaussian", WINNER_STRUCTURES,
)

offset_sensitivity = effect_table(offset_fits, winner_sign)
n_passing = int((offset_sensitivity["same sign"] & offset_sensitivity["significant"]).sum())
print(f"{n_passing} / {len(offset_sensitivity)} outcomes keep the winner's sign and a significant sharing-by-time coefficient")
offset_sensitivity

#### Sensitivity to the Thin Tail
The last year in the data has very few articles (see *The Data*): only the oldest cohort reaches it, and few of its articles share data. We keep it, since a mixed model gives so few observations little weight and the reviewer asked us not to trim the sample, but we check that the sharing-by-time coefficient does not depend on it by refitting the winner without that year.

In [ ]:
last_k = MODEL_DATA["k"].max()
tail = MODEL_DATA[MODEL_DATA["k"] == last_k]
print(f"k = {last_k} has {len(tail)} article-years ({int(tail['is_sharing_data'].sum())} sharing)")

tail_fits = {
    f"all years (k up to {last_k})": (winner_fit, WINNER_STRUCTURE),
    f"without k = {last_k}": fit_with_fallback(MODEL_DATA[MODEL_DATA["k"] < last_k], PRIMARY_OUTCOME, WINNER, PRIMARY_FAMILY, WINNER_STRUCTURES),
}
tail_sensitivity = effect_table(tail_fits, winner_sign)
tail_sensitivity

### Robustness Check #2: The Family
We refit the winner, and the first near-tied candidate if there is one (to keep the number of fits down), as a Gaussian on the raw counts, a Poisson and a negative binomial. Each is fitted with the full random-effects structure and falls back to random intercepts if it does not converge or is singular.<br>
The families do not estimate the same quantity (a Gaussian on `log(1 + y)` estimates a proportional change in the geometric mean, Poisson and negative binomial a rate ratio of the arithmetic mean), so we compare them on the sign and significance of the sharing-by-time coefficient, not on its size. The negative-binomial dispersion `theta` is estimated once, for the winner with random intercepts (`glmer.nb`, the slow fit), and then held fixed. AIC is only comparable between Poisson and negative binomial with the same `g(k)` and the same random structure.

In [ ]:
FAMILY_CANDIDATES = [WINNER] + [name for name in competitive if name != WINNER][:1]
FAMILIES = {            # label: (R family, outcome)
    "gaussian (raw)": ("gaussian", "citations"),
    "poisson": ("poisson", "citations"),
    "negative binomial": ("nb", "citations"),
}

nb_reference = fit_glmm(MODEL_DATA, build_formula("citations", WINNER, "intercept"), "nb")      # glmer.nb estimates theta: minutes
NB_THETA = nb_reference.fit["theta"]

family_fits = {}
for candidate in FAMILY_CANDIDATES:
    for label, (family, outcome) in FAMILIES.items():
        theta = {"theta": NB_THETA} if family == "nb" else {}
        family_fits[f"{candidate} | {label}"] = fit_with_fallback(MODEL_DATA, outcome, candidate, family, **theta)

family_comparison = effect_table(family_fits, winner_sign)
family_comparison["AIC"] = [
    fit.fit["aic"] if label.endswith(("poisson", "negative binomial")) else np.nan for label, (fit, _) in family_fits.items()
]
n_passing = int((family_comparison["same sign"] & family_comparison["significant"]).sum())
print(f"{n_passing} / {len(family_comparison)} fits keep the winner's sign and a significant sharing-by-time coefficient")
for candidate in FAMILY_CANDIDATES:
    (poisson_fit, poisson_structure), (nb_fit, nb_structure) = family_fits[f"{candidate} | poisson"], family_fits[f"{candidate} | negative binomial"]
    if poisson_structure != nb_structure:
        print(f"{candidate}: Poisson and negative binomial use different random structures, so their AICs are not comparable")
        continue
    better = "negative binomial" if nb_fit.fit["aic"] < poisson_fit.fit["aic"] else "Poisson"
    print(f"{candidate}: {better} has the lower AIC (by {abs(nb_fit.fit['aic'] - poisson_fit.fit['aic']):.1f}); Poisson overdispersion ratio {poisson_fit.fit['overdispersion']:.2f}")
family_comparison

### Visualize Model Predictions
We use the winner of the model selection to predict the expected citation counts per year for sharing vs. non-sharing articles, and plot the predicted citation dynamics.<br>
The predictions are for a typical article (covariates at their mean or mode, random effects at zero), for every year in the data (there is no minimum number of articles per year, so that no cutoff has to be defended). For the `log(1 + y)` model they are back-transformed to the geometric-mean scale (geometric mean of `1 + y`, minus 1), so they sit below the arithmetic mean of the counts. The points are model predictions, not observed means: the late years rest on few articles (see the *n* labels), and the downward bend at the end is partly the shape of the time term.<br>
`SHOW_ERROR_BANDS` toggles the 95% confidence band around each curve, `SHOW_ERROR_BARS` toggles +/- 1 SEM bars at each point (the standard error of the model's prediction, which is asymmetric on the back-transformed scale), and `SHOW_SAMPLE_SIZES` toggles the number of articles behind each point (*n*), written beside the point in the group's color: above it for the group with the higher prediction at that `k`, below it for the other, so the two labels never meet.


In [ ]:
SHOW_ERROR_BANDS = False    # the 95% confidence band around each curve
SHOW_ERROR_BARS = True      # +/- 1 SEM (the standard error of the model's prediction) at each point
SHOW_SAMPLE_SIZES = True    # the number of articles behind each point

predictions = winner_fit.predictions.copy()
if PRIMARY_OUTCOME == "log_citations":      # back-transform from log(1 + y): the geometric-mean scale, not the arithmetic mean
    interval_columns = ["fit", "ci_low", "ci_high", "sem_low", "sem_high"]
    predictions[interval_columns] = np.expm1(predictions[interval_columns])

dynamics_fig = go.Figure()
for is_sharing, label in ((1, "SHARING"), (0, "NOT SHARING")):
    subset = predictions[predictions["is_sharing_data"] == is_sharing]
    color = CLASS_COLORS[label]
    if SHOW_ERROR_BANDS:
        dynamics_fig.add_trace(go.Scatter(              # confidence band
            x=pd.concat([subset["k"], subset["k"][::-1]]),
            y=pd.concat([subset["ci_high"], subset["ci_low"][::-1]]),
            mode="lines", fill="toself", fillcolor=color, opacity=0.2, line=dict(width=0),
            hoverinfo="skip", showlegend=False, legendgroup=label,
        ))
    dynamics_fig.add_trace(go.Scatter(
        x=subset["k"], y=subset["fit"], name=label.title(), legendgroup=label,
        mode="lines+markers", line=dict(color=color), marker=dict(color=color),
        error_y=dict(                                   # asymmetric after the back-transform
            type="data", symmetric=False, visible=SHOW_ERROR_BARS, color=color, thickness=1.5, width=4,
            array=subset["sem_high"] - subset["fit"], arrayminus=subset["fit"] - subset["sem_low"],
        ),
    ))
    if SHOW_SAMPLE_SIZES:       # to the right of the dot, above it for the higher group and below it for the lower one
        other_fit = predictions[predictions["is_sharing_data"] == 1 - is_sharing].set_index("k")["fit"]
        next_fit = subset["fit"].shift(-1)
        for k, fit, following in zip(subset["k"], subset["fit"], next_fit):
            is_above = fit >= other_fit[k]
            is_line_in_the_way = following > fit if is_above else following < fit      # the curve runs toward the label
            offset = 12 if is_line_in_the_way else 4
            dynamics_fig.add_annotation(
                x=k, y=fit, text=f"<i>n</i>={sample_sizes[(is_sharing, int(k))]}", showarrow=False,
                font={**AXIS_TICK_FONT, "size": 12, "color": color},
                xanchor="left", xshift=8, yanchor="bottom" if is_above else "top", yshift=offset if is_above else -offset,
            )

# update layout and styling
dynamics_fig.update_xaxes(
    title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
    tickfont=AXIS_TICK_FONT, tickmode="linear", dtick=1,
)
dynamics_fig.update_yaxes(
    title=dict(text="Predicted Citations per Year (geometric mean)", font=AXIS_TITLE_FONT, standoff=15),
    tickfont=AXIS_TICK_FONT, rangemode="tozero",
)
dynamics_fig.update_layout(
    width=800, height=400,
    title=dict(text="Citation Dynamics: Sharing vs. Non-Sharing", font=TITLE_FONT, x=0.5, xanchor="center"),
    legend=dict(                    # horizontal, centred along the bottom of the plot area, just above the x axis
        orientation="h", font=LEGEND_FONT,
        xref="paper", x=0.5, xanchor="center", yref="paper", y=0.0, yanchor="bottom",
    ),
    margin=dict(t=60, b=0, l=0, r=0),
    template="plotly_white",
)

if False:
    dynamics_fig.show()

In [14]:
# flip to True to export this figure into `output/`
if False:
    save_figure(dynamics_fig, "citation_dynamics_model.png", width=800, height=400)

## Sharing Granularity
Not implemented yet: the winner's `g(k)` and random structure with `C(Sharing Class) * k` on the 80 sharing articles, to test whether the citation dynamics differ between sharing granularities (no model search, to keep the number of fits down).

In [15]:
1/0     # todo: drop this when working on sharing granularities

ZeroDivisionError: division by zero

---

## Sharing Granularity

In [ ]:
GROUPING_VARIABLE = "sharing_class"

DYNAMICS_GRANULARITY_DF = DYNAMICS_DF.loc[DYNAMICS_DF[GROUPING_VARIABLE] != "NONE"]
DYNAMICS_GRANULARITY_DF[GROUPING_VARIABLE] = DYNAMICS_GRANULARITY_DF[GROUPING_VARIABLE].cat.remove_unused_categories()

# verify we are left only with the three sharing classes
DYNAMICS_GRANULARITY_DF[GROUPING_VARIABLE].unique()

In [ ]:
# TODO

In [ ]:
# TODO: drop this to see the (deprecated) residuals analysis
1/0

---

## Citation Dynamics :: Univariate Comparisons (SUPERSEDED)
**This section is superseded and does not appear in the revised manuscript.** It is the analysis
reported in the submitted version, kept so the published v1 results stay reproducible. Everything
from this heading to the end of the notebook belongs to it, including the per-group comparisons at
the bottom.
<br><br>
We explor how citations differ **over time** between sharing and non-sharing articles; and whether different types of data sharing are associated with different citation dynamics.<br>
To do so, we calculate the number of citations in each calendar year post-publication for each article - ignoring "year 0" and the current year (2026) due to partial year effects. We then examine the differences in year-by-year citations across sharing categories (*sharing vs. non-sharing* and *sharing granularities*). We use a set of Mann-Whitney U-tests to compare the distributions of citations per year between groups, and correct for multiple comparisons using the Benjamini-Hochberg FDR procedure.

### (1) Sharing vs. Non-Sharing Articles
We compare the cumulative citation counts per year between articles that share data vs. those that do not.

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=True
)
# drop years with fewer than 20 articles combined
cit_dynamics_sharing = cit_dynamics_sharing.loc[:, cit_dynamics_sharing.count(axis=0) >= 20]

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    non_sharing_is_normal = stats.shapiro(non_sharing_vals).pvalue > 0.05
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    sharing_is_normal = stats.shapiro(sharing_vals).pvalue > 0.05
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    yearly_ttest = pg.ttest(non_sharing_vals, sharing_vals, paired=False, alternative="less", correction=True)
    yearly_mwtest = pg.mwu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = yearly_mwtest.loc["MWU", "U_val"]
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_is_normal": non_sharing_is_normal,
        "non_sharing_median": non_sharing_vals.median(),
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),

        "sharing_count": n2,
        "sharing_is_normal": sharing_is_normal,
        "sharing_median": sharing_vals.median(),
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),

        "t_stat": yearly_ttest.loc["T_test", "T"],
        "t_dof": yearly_ttest.loc["T_test", "dof"],
        "t_p_raw": yearly_ttest.loc["T_test", "p_val"],
        "t_cohen_d": yearly_ttest.loc["T_test", "cohen_d"],

        "U_nonshare": U_nonshare,
        "U_share": U_share,
        "mw_p_raw": yearly_mwtest.loc["MWU", "p_val"],
        "r_rb": abs(yearly_mwtest.loc["MWU", "RBC"]),
        "cles": yearly_mwtest.loc["MWU", "CLES"],
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
t_reject, t_p_corrected, _, _ = multipletests(year_stats['t_p_raw'], alpha=0.05, method='fdr_bh')
mw_reject, mw_p_corrected, _, _ = multipletests(year_stats['mw_p_raw'], alpha=0.05, method='fdr_bh')
year_stats['t_p_corrected'] = t_p_corrected
year_stats['t_is_significant'] = t_reject
year_stats['mw_p_corrected'] = mw_p_corrected
year_stats['mw_is_significant'] = mw_reject
column_order = [
    "year",
    "non_sharing_is_normal", "sharing_is_normal",
    "t_is_significant", "t_p_corrected",
    "mw_is_significant", "mw_p_corrected",
    "t_stat", "t_dof", "t_p_raw", "t_cohen_d",
    "U_share", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem", "non_sharing_median",
    "sharing_count", "sharing_mean", "sharing_sem", "sharing_median",
]
year_stats = year_stats[column_order]

year_stats

In [ ]:
LINE_ANNOTATION_X = 3

cit_dynamics_sharing_fig = go.Figure()
for is_share in cit_dynamics_sharing.index.get_level_values(GROUPING_COL).unique().sort_values():
    subset = cit_dynamics_sharing.xs(is_share, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    label = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[label.upper()]
    cit_dynamics_sharing_fig.add_trace(go.Scatter(
        name=label, legendgroup=label,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=color,
            # size=np.sqrt(count_per_year.values),
        ),
        mode='lines+markers',
    ))
    # annotate the lines
    if is_share:
        ann_y = mean_per_year[LINE_ANNOTATION_X] + 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "right"
        yanchor = "bottom"
    else:
        ann_y = mean_per_year[LINE_ANNOTATION_X] - 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "left"
        yanchor = "top"
    cit_dynamics_sharing_fig.add_annotation(
        x=LINE_ANNOTATION_X, xanchor=xanchor, xref="x",
        y=ann_y, yanchor=yanchor, yref="y",
        text=label, font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )

# add significance asterisks
SIGNIFICANCE_ANNOTATION_Y = 40
for _, row in year_stats.iterrows():
    if row["mw_p_corrected"] >= 0.1:
        continue
    fontsize = 22
    yanchor = "middle"
    if row["mw_p_corrected"] < 0.001:
        ann_text = "<b>***</b>"
    elif row["mw_p_corrected"] < 0.01:
        ann_text = "<b>**</b>"
    elif row["mw_p_corrected"] < 0.05:
        ann_text = "<b>*</b>"
    else:
        ann_text = "\u2020"     # dagger for p < 0.
        fontsize = 16
        yanchor = "bottom"
    cit_dynamics_sharing_fig.add_annotation(
        x=row["year"], xanchor="center", xref="x",
        y=SIGNIFICANCE_ANNOTATION_Y, yanchor=yanchor, yref="y",
        text=ann_text, font={**TITLE_FONT, "size": fontsize},
        showarrow=False,
    )

cit_dynamics_sharing_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Sharing vs. Non-Sharing Articles",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=1.0, xanchor="right", y=0.4, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
        visible=False,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_fig, "citation_dynamics_sharing_vs_nonsharing.png", width=800, height=400)

##### Source of the Difference
So we found a difference in yearly cumulative citations between sharing and non-sharing articles.<br>
But where does this difference come from? Is it because the sharing articles get cited more frequently in each year (i.e., higher "velocity" of citations)? Or is it because they got an initian citation boost which then accumulates over time (i.e., higher "stock" of citations)?<br>
To answer this question, we repeat the above analysis but now looking at **non-cumulative** citation counts per year (i.e., the "velocity" of citations).

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=False
)

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    year_pair_test = stats.mannwhitneyu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = year_pair_test.statistic
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),
        "sharing_count": n2,
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),
        "U_nonshare": U_nonshare,
        "p_raw": year_pair_test.pvalue,
        "r_rb": rank_biserial_corr,
        "cles": cles,
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "year",
    "is_significant", "p_corrected", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem",
    "sharing_count", "sharing_mean", "sharing_sem",
    "p_raw"
]
year_stats = year_stats[column_order]

year_stats

### (2) Different Types of Data Sharing

In [ ]:
GROUPING_COL = "data_sharing_class"

cit_dynamics_sharing_class = citations_per_year(
    combined.loc[combined["is_sharing_data"]], grouping_col=GROUPING_COL
)

year_stats = dict()
for year in cit_dynamics_sharing_class.columns:
    year_stats[year] = dict()
    class_values = []
    for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
        subset = (cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)[year]).dropna()
        if subset.empty:
            continue
        year_stats[year][f"{share_class}_count"] = subset.count()
        year_stats[year][f"{share_class}_mean"] = subset.mean()
        year_stats[year][f"{share_class}_sem"] = subset.sem()
        class_values.append(subset)
    sharegroup_test = stats.kruskal(*class_values)
    year_stats[year]["W"] = sharegroup_test.statistic
    year_stats[year]["p_raw"] = sharegroup_test.pvalue

year_stats = pd.DataFrame.from_dict(year_stats, orient='index')
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "is_significant", "p_corrected", "W", "p_raw",
    *[f"{share_class}_{stat}" for share_class in ["FIXATION", "TRIAL", "PARTICIPANT"] for stat in ["count", "mean", "sem"]]
]
year_stats = year_stats[column_order]
year_stats

In [ ]:
cit_dynamics_sharing_class_fig = go.Figure()
for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
    subset = cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    cit_dynamics_sharing_class_fig.add_trace(go.Scatter(
        name=share_class, legendgroup=share_class,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=CLASS_COLORS[share_class],
            size=count_per_year.values
        ),
        mode='lines+markers',
    ))

cit_dynamics_sharing_class_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Different Types of Data Sharing",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_class_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_class_fig, "citation_dynamics_sharing_class.png", width=800, height=400)